# CNN + MOGEO + AdamW Training -- Medicinal Plant Classifier

This notebook runs the full pipeline on a Colab GPU, with all datasets, checkpoints,
and results persisted to Google Drive so training survives Colab disconnects.

**Pipeline stages (run cells top to bottom, in order):**
1. GPU check
2. Mount Google Drive
3. Clone/update the repository (into Drive, so it persists)
4. Install dependencies
5. Verify dataset + build the leakage-safe 85%/15% split manifest
6. Baseline training (fixed hyperparameters, AdamW)
7. MOGEO hyperparameter search (resumable, ~10 epochs/candidate)
8. Final training with MOGEO-selected hyperparameters (max 40 epochs, patience 7) -> `best.pt`
9. One-time final evaluation on the untouched 15% test split -> reports + plots
10. Download `best.pt` + results back to your machine

**Rule enforced throughout:** the final 15% test split (`final_test`) is only ever
touched by `evaluate_final.py`, once, at the very end. See `instructions.md` for the
full step-by-step reference (this notebook mirrors it).

## 1. GPU check
Go to **Runtime -> Change runtime type -> T4 GPU** (or better) before running this cell.

In [ ]:
!nvidia-smi


## 2. Mount Google Drive
This is what makes the dataset, checkpoints, and results survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone (or update) the repository into Drive
The repo is cloned **into your Drive**, not into the ephemeral Colab VM disk, so
the working tree (dataset + code + checkpoints you generate) persists across sessions.
Re-running this cell in a later session simply `git pull`s the latest code.

In [ ]:
import os

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/cnn_clasifier_project'
REPO_URL = 'https://github.com/Arnav131/cnn_clasifier.git'
REPO_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'cnn_clasifier')

os.makedirs(DRIVE_PROJECT_ROOT, exist_ok=True)

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Cloning repository into Drive (first run) ...')
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    print('Repository already present -- pulling latest changes ...')
    !cd "{REPO_DIR}" && git pull

# The repository root itself is named "code" inside this checkout in some
# setups -- verify the actual project folder that contains dataset/, pipeline/, etc.
print('\nRepo contents:')
!ls "{REPO_DIR}"


In [ ]:
# Set PROJECT_DIR to the folder that actually contains dataset/, pipeline/,
# run_baseline.py, etc. Adjust this if your repo layout differs.
PROJECT_DIR = REPO_DIR
if not os.path.exists(os.path.join(PROJECT_DIR, 'pipeline')):
    # common case: the pipeline code lives inside a "code" subfolder
    candidate = os.path.join(REPO_DIR, 'code')
    if os.path.exists(os.path.join(candidate, 'pipeline')):
        PROJECT_DIR = candidate

print('PROJECT_DIR =', PROJECT_DIR)
assert os.path.exists(os.path.join(PROJECT_DIR, 'pipeline')), (
    'Could not locate the pipeline/ package. Check REPO_DIR / PROJECT_DIR above.'
)
%cd {PROJECT_DIR}


## 4. Install dependencies
`torch`/`torchvision` are already GPU-enabled in the Colab runtime and are
intentionally **not** reinstalled here (see `requirements-colab.txt` for why).

In [ ]:
!pip install -q -r requirements-colab.txt

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is not available -- check Runtime > Change runtime type > GPU.'


## 5. Dataset verification
Confirms the expected ~150 classes / ~5250 images are present before doing anything else.

In [ ]:
import os

dataset_dir = 'dataset'
classes = sorted(d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d)))
counts = {c: len(os.listdir(os.path.join(dataset_dir, c))) for c in classes}
total = sum(counts.values())

print(f'classes found: {len(classes)}')
print(f'total images: {total}')
print(f'min/max/avg per class: {min(counts.values())} / {max(counts.values())} / {total/len(classes):.1f}')

assert len(classes) == 150, f'expected 150 classes, found {len(classes)}'
assert total > 4000, f'suspiciously few images found ({total}) -- did Drive/clone finish syncing?'


## 6. Build the train/test split + run leakage checks
This is done **once**. It writes `splits/manifest.csv` (dev_train / dev_val / final_test
assignment for every image, fixed by a seeded stratified split) and `splits/classes.json`.
Every later script (baseline, MOGEO, final training, evaluation) reads this SAME manifest,
so `final_test` is guaranteed identical and untouched everywhere. It also detects and
resolves any byte-identical duplicate images that would otherwise leak across splits.

In [ ]:
from pipeline.data import build_manifest
from pipeline.utils import leakage_report

df = build_manifest('dataset', 'splits')  # cached after first run -- safe to re-run
print(df['split'].value_counts())
print()
print(leakage_report(df.to_dict('records')))

with open('splits/duplicate_report.txt') as f:
    print()
    print(f.read())


## 7. Baseline experiment
Fixed hyperparameters (architecture close in spirit to the historical model), trained
with AdamW through the corrected pipeline. This isolates "fixing the pipeline" from
"adding MOGEO search" so the two contributions can be compared honestly.

If this cell is interrupted (Colab disconnect), just re-run it -- `--resume` picks the
run back up from its last saved epoch checkpoint on Drive.

In [ ]:
!python run_baseline.py --dataset-dir dataset --output-dir . --resume


## 8. MOGEO hyperparameter/architecture search
A genuine Multi-Objective Golden Eagle Optimizer (attack/cruise flight vectors,
Pareto-archive with non-dominated sorting + crowding distance -- see `pipeline/mogeo.py`)
searches CNN architecture + AdamW hyperparameters. Each candidate trains for only
~10 epochs on `dev_train`, evaluated on `dev_val`. `final_test` is never touched here.

**Default budget:** population=8, generations=6 (<= 48 short trainings total).
Increase `--pop-size` / `--generations` if you have more GPU time available.

**Resumability:** MOGEO state is checkpointed to Drive after *every single candidate*.
If Colab disconnects mid-search, just re-run this exact cell -- it continues from the
last completed candidate instead of restarting the whole search.

In [ ]:
!python run_mogeo.py --dataset-dir dataset --output-dir . --pop-size 8 --generations 6 --resume


### Inspect the MOGEO convergence plot and selected hyperparameters

In [ ]:
import json
from IPython.display import Image, display

display(Image(filename='results/mogeo_convergence.png'))

with open('results/best_hparams.json') as f:
    print(json.dumps(json.load(f), indent=2))


## 9. Final training
Retrains with the MOGEO-selected hyperparameters on the full `dev_train`/`dev_val`
split. Hard cap of **40 epochs**, early stopping **patience = 7** -- training stops
as soon as validation accuracy plateaus, it does not blindly run to 40 epochs.
Produces `checkpoints/best.pt` (deliverable). Resumable the same way as above.

In [ ]:
!python run_final.py --dataset-dir dataset --output-dir . --resume


## 10. Final evaluation -- ONE TIME on the untouched final_test split
This is the only script in the whole pipeline that reads `final_test`. Run it once,
after final training has converged. It writes `accuracy_report.md`, `accuracy_report.json`,
`confusion_matrix.png`, and `error_analysis.md` to `results/`.

In [ ]:
!python evaluate_final.py --dataset-dir dataset --output-dir .


In [ ]:
from IPython.display import Image, Markdown, display

with open('results/accuracy_report.md') as f:
    display(Markdown(f.read()))
display(Image(filename='results/confusion_matrix.png'))


## 11. Download `best.pt` and results back to your machine
Zips the checkpoint + all report/plot artifacts and triggers a browser download.
Everything also remains on Drive at `PROJECT_DIR` for later use.

In [ ]:
import shutil
from google.colab import files

bundle_name = 'cnn_clasifier_deliverables'
shutil.make_archive(bundle_name, 'zip', '.', 'checkpoints')
shutil.make_archive(bundle_name + '_results', 'zip', '.', 'results')

files.download(bundle_name + '.zip')
files.download(bundle_name + '_results.zip')


## 12. Feed `best.pt` back to Antigravity / test locally
After downloading, place `best.pt` at `checkpoints/best.pt` in your local clone of this
repository, then run (locally, or in this Colab session):

```bash
python test_model.py --checkpoint checkpoints/best.pt --image path/to/some_leaf.jpg
python test_model.py --checkpoint checkpoints/best.pt --eval-test --dataset-dir dataset
```

See `instructions.md` for the full walkthrough, including how to commit the generated
results/checkpoints back to the GitHub repository if desired.